In [2]:
import os

# 目标文件夹路径（改成你要检查的目录）
folder_path = "/home/jannik/Documents/mri_mat_onehot"

# 遍历并打印所有文件名
for root, dirs, files in os.walk(folder_path):
    for file_name in files:
        print(os.path.join(root, file_name))


/home/jannik/Documents/mri_mat_onehot/OHC_13_ckgulxe.mat
/home/jannik/Documents/mri_mat_onehot/OHC_16_degkbxy.mat
/home/jannik/Documents/mri_mat_onehot/YHC_04_nnvuqjv.mat
/home/jannik/Documents/mri_mat_onehot/OHC_07_xhttjob.mat
/home/jannik/Documents/mri_mat_onehot/ODP_02_jfmiyxb.mat
/home/jannik/Documents/mri_mat_onehot/YHC_02_terejqu.mat
/home/jannik/Documents/mri_mat_onehot/PDP_12_vubknkr.mat
/home/jannik/Documents/mri_mat_onehot/YHC_10_lncguay.mat
/home/jannik/Documents/mri_mat_onehot/OHC_18_kgkcdyy.mat
/home/jannik/Documents/mri_mat_onehot/PDP_04_irdscqm.mat
/home/jannik/Documents/mri_mat_onehot/YHC_08_xzlcgkp.mat
/home/jannik/Documents/mri_mat_onehot/PDP_01_smpglzo.mat
/home/jannik/Documents/mri_mat_onehot/OHC_06_gqmjvpo.mat
/home/jannik/Documents/mri_mat_onehot/OHC_09_aieuine.mat
/home/jannik/Documents/mri_mat_onehot/YHC_01_eqtqloh.mat
/home/jannik/Documents/mri_mat_onehot/OHC_10_pgrrtlc.mat
/home/jannik/Documents/mri_mat_onehot/YHC_05_njywoqp.mat
/home/jannik/Documents/mri_mat_

In [3]:
import h5py
import numpy as np

# 检查TRAIN38.mat的结构
f = h5py.File("/home/jannik/Documents/TRAIN38.mat",'r')
print("TRAIN38.mat中的变量:")
for k, v in f.items():
    print(f"{k}: shape = {v.shape}, dtype = {v.dtype}")

# 检查prob_idx的唯一值
prob_idx = np.array(f['prob_idx']).transpose()
print(f"\nprob_idx唯一值: {np.unique(prob_idx)}")
print(f"prob_idx范围: {prob_idx.min()} - {prob_idx.max()}")

# 检查region标签
region = np.array(f['region']).transpose()
print(f"\nregion shape: {region.shape}")
print(f"region中的唯一类别数: {np.sum(np.any(region, axis=0))}")

f.close()

TRAIN38.mat中的变量:
all_age: shape = (1, 6968700), dtype = float64
data: shape = (341, 6968700), dtype = float32
prob_idx: shape = (1, 6968700), dtype = float64
region: shape = (102, 6968700), dtype = uint8

prob_idx唯一值: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18.
 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 34. 35. 36.
 37. 38.]
prob_idx范围: 1.0 - 38.0

region shape: (6968700, 102)
region中的唯一类别数: 101


In [6]:
import h5py
import numpy as np
import os
import pandas as pd
from pathlib import Path

def analyze_single_mat_file(filepath):
    """分析单个MAT文件的结构"""
    try:
        with h5py.File(filepath, 'r') as f:
            file_info = {
                'filename': os.path.basename(filepath),
                'variables': {},
                'total_size_mb': 0
            }
            
            print(f"\n=== 分析文件: {file_info['filename']} ===")
            
            # 分析每个变量
            for key, value in f.items():
                var_info = {
                    'shape': value.shape,
                    'dtype': str(value.dtype),
                    'size_mb': value.nbytes / (1024 * 1024)
                }
                file_info['variables'][key] = var_info
                file_info['total_size_mb'] += var_info['size_mb']
                
                print(f"  {key}: shape={value.shape}, dtype={value.dtype}, size={var_info['size_mb']:.2f}MB")
                
                # 如果是小数组，显示一些统计信息
                if value.size < 1000000:  # 小于1M个元素
                    try:
                        data = np.array(value)
                        if data.dtype in [np.float32, np.float64, np.int32, np.int64]:
                            print(f"    统计: min={np.min(data):.4f}, max={np.max(data):.4f}, mean={np.mean(data):.4f}")
                            if data.ndim == 1 and data.size < 100:
                                print(f"    前10个值: {data[:10]}")
                    except:
                        print(f"    无法读取统计信息")
            
            print(f"  总文件大小: {file_info['total_size_mb']:.2f}MB")
            return file_info
            
    except Exception as e:
        print(f"读取文件 {filepath} 时出错: {e}")
        return None

def analyze_all_mat_files():
    """分析所有38个MAT文件"""
    
    # 文件路径列表
    file_paths = [
        "/home/jannik/Documents/mri_mat_onehot/OHC_13_ckgulxe.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_16_degkbxy.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_04_nnvuqjv.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_07_xhttjob.mat",
        "/home/jannik/Documents/mri_mat_onehot/ODP_02_jfmiyxb.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_02_terejqu.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_12_vubknkr.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_10_lncguay.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_18_kgkcdyy.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_04_irdscqm.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_08_xzlcgkp.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_01_smpglzo.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_06_gqmjvpo.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_09_aieuine.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_01_eqtqloh.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_10_pgrrtlc.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_05_njywoqp.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_06_atnmxpq.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_17_mmixjcu.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_08_mezsnmb.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_05_vgvgyjf.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_03_yipftgn.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_01_tuemlqs.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_11_fxdvnzc.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_09_rbbncvv.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_05_usxxisf.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_15_wqjnkbg.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_19_obgjvab.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_10_sdrnzqu.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_03_cyumllh.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_02_thrredm.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_06_fhmfvff.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_07_wuoegxa.mat",
        "/home/jannik/Documents/mri_mat_onehot/ODP_01_qhlazec.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_03_zaxxucg.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_02_ngmqrkj.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_14_gddwpod.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_11_llvcdek.mat"
    ]
    
    all_file_info = []
    
    print("开始分析38个MAT文件...")
    
    for filepath in file_paths:
        if os.path.exists(filepath):
            file_info = analyze_single_mat_file(filepath)
            if file_info:
                all_file_info.append(file_info)
        else:
            print(f"文件不存在: {filepath}")
    
    # 总结分析
    print("\n" + "="*80)
    print("总结分析:")
    print("="*80)
    
    if all_file_info:
        # 统计变量名
        all_variables = set()
        for info in all_file_info:
            all_variables.update(info['variables'].keys())
        
        print(f"成功分析了 {len(all_file_info)} 个文件")
        print(f"发现的变量名: {sorted(list(all_variables))}")
        
        # 检查每个变量在所有文件中的一致性
        for var_name in sorted(all_variables):
            shapes = []
            dtypes = []
            for info in all_file_info:
                if var_name in info['variables']:
                    shapes.append(info['variables'][var_name]['shape'])
                    dtypes.append(info['variables'][var_name]['dtype'])
            
            print(f"\n变量 '{var_name}':")
            print(f"  出现在 {len(shapes)}/{len(all_file_info)} 个文件中")
            if shapes:
                unique_shapes = list(set([str(s) for s in shapes]))
                unique_dtypes = list(set(dtypes))
                print(f"  形状: {unique_shapes}")
                print(f"  数据类型: {unique_dtypes}")
        
        # 平均文件大小
        avg_size = np.mean([info['total_size_mb'] for info in all_file_info])
        print(f"\n平均文件大小: {avg_size:.2f}MB")
    
    return all_file_info

def create_mapping_table():
    """创建文件名到prob_idx的映射表"""
    print("\n" + "="*80)
    print("创建文件名到prob_idx的映射表:")
    print("="*80)
    
    # 从文件名提取信息
    mapping_data = []
    
    file_paths = [
        "/home/jannik/Documents/mri_mat_onehot/OHC_13_ckgulxe.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_16_degkbxy.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_04_nnvuqjv.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_07_xhttjob.mat",
        "/home/jannik/Documents/mri_mat_onehot/ODP_02_jfmiyxb.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_02_terejqu.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_12_vubknkr.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_10_lncguay.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_18_kgkcdyy.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_04_irdscqm.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_08_xzlcgkp.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_01_smpglzo.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_06_gqmjvpo.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_09_aieuine.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_01_eqtqloh.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_10_pgrrtlc.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_05_njywoqp.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_06_atnmxpq.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_17_mmixjcu.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_08_mezsnmb.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_05_vgvgyjf.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_03_yipftgn.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_01_tuemlqs.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_11_fxdvnzc.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_09_rbbncvv.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_05_usxxisf.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_15_wqjnkbg.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_19_obgjvab.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_10_sdrnzqu.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_03_cyumllh.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_02_thrredm.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_06_fhmfvff.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_07_wuoegxa.mat",
        "/home/jannik/Documents/mri_mat_onehot/ODP_01_qhlazec.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_03_zaxxucg.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_02_ngmqrkj.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_14_gddwpod.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_11_llvcdek.mat"
    ]
    
    for i, filepath in enumerate(file_paths, 1):
        filename = os.path.basename(filepath)
        # 解析文件名：group_number_code.mat
        parts = filename.replace('.mat', '').split('_')
        if len(parts) >= 3:
            group = parts[0]
            number = parts[1]
            code = parts[2]
            
            mapping_data.append({
                'prob_idx': i,
                'filename': filename,
                'filepath': filepath,
                'group': group,
                'number': int(number),
                'code': code,
                'exists': os.path.exists(filepath)
            })
    
    # 创建DataFrame并显示
    df = pd.DataFrame(mapping_data)
    print(df.to_string(index=False))
    
    return df

if __name__ == "__main__":
    # 分析所有文件
    file_analysis = analyze_all_mat_files()
    
    # 创建映射表
    mapping_df = create_mapping_table()
    
    print(f"\n处理完成！分析了 {len(file_analysis)} 个文件。")

开始分析38个MAT文件...

=== 分析文件: OHC_13_ckgulxe.mat ===
  big_seg: shape=(384, 336, 256), dtype=float64, size=252.00MB
  multidim_data: shape=(351, 2069287), dtype=float32, size=2770.69MB
  region: shape=(384, 336, 256), dtype=uint8, size=31.50MB
  region_seg: shape=(1, 2069287), dtype=float64, size=15.79MB
  seg_one_hot: shape=(102, 2069287), dtype=uint8, size=201.29MB
  总文件大小: 3271.27MB

=== 分析文件: OHC_16_degkbxy.mat ===
  big_seg: shape=(384, 336, 256), dtype=float64, size=252.00MB
  multidim_data: shape=(351, 1850427), dtype=float32, size=2477.65MB
  region: shape=(384, 336, 256), dtype=uint8, size=31.50MB
  region_seg: shape=(1, 1850427), dtype=float64, size=14.12MB
  seg_one_hot: shape=(102, 1850427), dtype=uint8, size=180.00MB
  总文件大小: 2955.26MB

=== 分析文件: YHC_04_nnvuqjv.mat ===
  big_seg: shape=(384, 336, 256), dtype=float64, size=252.00MB
  multidim_data: shape=(351, 2105627), dtype=float32, size=2819.35MB
  region: shape=(384, 336, 256), dtype=uint8, size=31.50MB
  region_seg: shape

In [7]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

def analyze_train38_structure():
    """深度分析TRAIN38.mat的数据结构"""
    
    print("="*80)
    print("TRAIN38.mat 深度分析")
    print("="*80)
    
    with h5py.File('TRAIN38.mat', 'r') as f:
        # 加载所有数据
        print("正在加载数据...")
        data = np.array(f['data']).T  # (6968700, 341)
        region = np.array(f['region']).T  # (6968700, 102)
        prob_idx = np.array(f['prob_idx']).T.flatten()  # (6968700,)
        all_age = np.array(f['all_age']).T.flatten()  # (6968700,)
        
        print(f"数据加载完成:")
        print(f"  data shape: {data.shape}")
        print(f"  region shape: {region.shape}")
        print(f"  prob_idx shape: {prob_idx.shape}")
        print(f"  all_age shape: {all_age.shape}")
        
        # 分析每个被试的数据分布
        print(f"\n每个被试的数据点数量:")
        prob_counts = Counter(prob_idx)
        for prob_id in sorted(prob_counts.keys()):
            count = prob_counts[prob_id]
            print(f"  被试 {int(prob_id):2d}: {count:8d} 个数据点")
        
        print(f"\n数据点数量统计:")
        counts = list(prob_counts.values())
        print(f"  总数据点: {sum(counts):,}")
        print(f"  平均每个被试: {np.mean(counts):,.0f}")
        print(f"  最少: {min(counts):,}")
        print(f"  最多: {max(counts):,}")
        print(f"  标准差: {np.std(counts):,.0f}")
        
        # 分析年龄信息
        print(f"\n年龄信息分析:")
        unique_ages = []
        for prob_id in sorted(prob_counts.keys()):
            mask = prob_idx == prob_id
            age_values = all_age[mask]
            unique_age = np.unique(age_values)
            if len(unique_age) == 1:
                unique_ages.append(unique_age[0])
                print(f"  被试 {int(prob_id):2d}: {unique_age[0]:.1f} 岁")
            else:
                print(f"  被试 {int(prob_id):2d}: 年龄不一致! {unique_age}")
        
        if unique_ages:
            print(f"\n年龄统计:")
            print(f"  平均年龄: {np.mean(unique_ages):.1f}")
            print(f"  年龄范围: {min(unique_ages):.1f} - {max(unique_ages):.1f}")
        
        # 分析region标签分布
        print(f"\nregion标签分析:")
        active_regions = np.sum(region, axis=0)  # 每个region的激活次数
        active_region_indices = np.where(active_regions > 0)[0]
        print(f"  激活的region数量: {len(active_region_indices)}/102")
        print(f"  最常激活的region: {np.argmax(active_regions)} (激活{int(np.max(active_regions))}次)")
        print(f"  最少激活的region: {active_region_indices[np.argmin(active_regions[active_region_indices])]} (激活{int(np.min(active_regions[active_regions > 0]))}次)")
        
        # 分析341维特征的特点
        print(f"\n341维特征分析:")
        print(f"  特征均值范围: {np.min(np.mean(data, axis=0)):.6f} - {np.max(np.mean(data, axis=0)):.6f}")
        print(f"  特征标准差范围: {np.min(np.std(data, axis=0)):.6f} - {np.max(np.std(data, axis=0)):.6f}")
        
        # 检查特征是否有缺失值或异常值
        nan_features = np.sum(np.isnan(data), axis=0)
        inf_features = np.sum(np.isinf(data), axis=0)
        print(f"  包含NaN的特征数: {np.sum(nan_features > 0)}")
        print(f"  包含Inf的特征数: {np.sum(inf_features > 0)}")
        
        return {
            'data': data,
            'region': region, 
            'prob_idx': prob_idx,
            'all_age': all_age,
            'prob_counts': prob_counts,
            'active_regions': active_regions
        }

def analyze_data_patterns(analysis_result):
    """分析数据模式和可能的3D结构"""
    
    print("\n" + "="*80)
    print("数据模式分析")
    print("="*80)
    
    data = analysis_result['data']
    prob_counts = analysis_result['prob_counts']
    
    # 分析数据点数量的可能来源
    print("推测3D数据结构:")
    
    # 常见的数据点数量
    unique_counts = list(set(prob_counts.values()))
    print(f"发现 {len(unique_counts)} 种不同的数据点数量:")
    
    for count in sorted(unique_counts):
        # 尝试因式分解，寻找可能的3D尺寸
        factors = []
        n = count
        
        # 寻找因子
        possible_dims = []
        for i in range(2, int(n**0.5) + 1):
            while n % i == 0:
                factors.append(i)
                n //= i
        if n > 1:
            factors.append(n)
        
        # 尝试组合成3D尺寸
        if len(factors) >= 3:
            # 尝试不同的组合
            from itertools import combinations
            for r in range(3, min(6, len(factors) + 1)):
                for combo in combinations(range(len(factors)), r):
                    dims = [1, 1, 1]
                    remaining_factors = factors.copy()
                    for idx in combo:
                        dims[0] *= factors[idx]
                    
                    # 剩余因子分配给其他维度
                    remaining = count // dims[0]
                    if remaining > 1:
                        # 简单的分解
                        sqrt_remaining = int(remaining**0.5)
                        if sqrt_remaining * sqrt_remaining == remaining:
                            dims[1] = dims[2] = sqrt_remaining
                        else:
                            for i in range(2, int(remaining**0.5) + 1):
                                if remaining % i == 0:
                                    dims[1] = i
                                    dims[2] = remaining // i
                                    break
                    
                    if dims[0] * dims[1] * dims[2] == count:
                        possible_dims.append(tuple(sorted(dims, reverse=True)))
        
        # 去重并显示
        possible_dims = list(set(possible_dims))
        prob_with_this_count = [p for p, c in prob_counts.items() if c == count]
        
        print(f"\n  {count:,} 个数据点 (被试: {[int(p) for p in prob_with_this_count]}):")
        print(f"    质因数分解: {factors}")
        if possible_dims:
            print(f"    可能的3D尺寸: {possible_dims[:5]}")  # 只显示前5个
        else:
            print(f"    无法分解为合理的3D尺寸")

def suggest_dataset_construction():
    """建议数据集构建方法"""
    
    print("\n" + "="*80)
    print("数据集构建建议")
    print("="*80)
    
    print("""
基于分析结果，我建议以下数据集构建方法：

1. **理解数据结构**:
   - TRAIN38.mat包含6,968,700个数据点，来自38个被试
   - 每个数据点有341维特征（多模态MRI特征）
   - 每个数据点对应102个脑区域中的一个
   - 数据点数量在不同被试间差异很大

2. **3D到特征的映射**:
   - 你的单独3D文件需要提取成341维特征
   - 这可能涉及：
     * 多模态配准（QTI, CEST, MPRAGE等）
     * 体素级特征提取
     * 空间归一化
     * 特征降维或选择

3. **构建新数据集的步骤**:
   
   步骤1: 分析你的3D文件结构
   步骤2: 实现特征提取管道
   步骤3: 生成region标签（freesurfer分割）
   步骤4: 组合成TRAIN38格式

4. **需要的信息**:
   - 你的3D文件包含哪些变量？
   - 3D数据的尺寸是什么？
   - 你有对应的freesurfer分割结果吗？
   - 如何从多模态3D数据提取341维特征？
""")

if __name__ == "__main__":
    # 执行分析
    print("开始分析TRAIN38.mat...")
    analysis_result = analyze_train38_structure()
    
    print("\n分析数据模式...")
    analyze_data_patterns(analysis_result)
    
    print("\n生成构建建议...")
    suggest_dataset_construction()

开始分析TRAIN38.mat...
TRAIN38.mat 深度分析
正在加载数据...
数据加载完成:
  data shape: (6968700, 341)
  region shape: (6968700, 102)
  prob_idx shape: (6968700,)
  all_age shape: (6968700,)

每个被试的数据点数量:
  被试  1:   181860 个数据点
  被试  2:   151377 个数据点
  被试  3:   188829 个数据点
  被试  4:   195992 个数据点
  被试  5:   174920 个数据点
  被试  6:   177934 个数据点
  被试  7:   181605 个数据点
  被试  8:   187188 个数据点
  被试  9:   152115 个数据点
  被试 10:   199420 个数据点
  被试 11:   186886 个数据点
  被试 12:   189364 个数据点
  被试 13:   169167 个数据点
  被试 14:   180501 个数据点
  被试 15:   169451 个数据点
  被试 16:   186332 个数据点
  被试 17:   168639 个数据点
  被试 18:   217218 个数据点
  被试 19:   213492 个数据点
  被试 20:   189540 个数据点
  被试 21:   220259 个数据点
  被试 22:   190482 个数据点
  被试 23:   215189 个数据点
  被试 24:   118219 个数据点
  被试 25:   201563 个数据点
  被试 26:   164818 个数据点
  被试 27:   165501 个数据点
  被试 28:   183237 个数据点
  被试 29:   192397 个数据点
  被试 30:   181606 个数据点
  被试 31:   197834 个数据点
  被试 32:   177014 个数据点
  被试 33:   196190 个数据点
  被试 34:   192669 个数据点
  被试 35:   180025 个数据点
  被试 36:   

In [8]:
import h5py
import numpy as np
import pandas as pd
from collections import Counter
import os

def analyze_segmentation_distributions():
    """
    分析38个MAT文件中big_seg、region_seg、region的数值分布
    """
    
    file_paths = [
        "/home/jannik/Documents/mri_mat_onehot/OHC_13_ckgulxe.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_16_degkbxy.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_04_nnvuqjv.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_07_xhttjob.mat",
        "/home/jannik/Documents/mri_mat_onehot/ODP_02_jfmiyxb.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_02_terejqu.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_12_vubknkr.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_10_lncguay.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_18_kgkcdyy.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_04_irdscqm.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_08_xzlcgkp.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_01_smpglzo.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_06_gqmjvpo.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_09_aieuine.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_01_eqtqloh.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_10_pgrrtlc.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_05_njywoqp.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_06_atnmxpq.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_17_mmixjcu.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_08_mezsnmb.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_05_vgvgyjf.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_03_yipftgn.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_01_tuemlqs.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_11_fxdvnzc.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_09_rbbncvv.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_05_usxxisf.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_15_wqjnkbg.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_19_obgjvab.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_10_sdrnzqu.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_03_cyumllh.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_02_thrredm.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_06_fhmfvff.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_07_wuoegxa.mat",
        "/home/jannik/Documents/mri_mat_onehot/ODP_01_qhlazec.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_03_zaxxucg.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_02_ngmqrkj.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_14_gddwpod.mat",
        "/home/jannik/Documents/mri_mat_onehot/PDP_11_llvcdek.mat"
    ]
    
    print("="*80)
    print("分析38个MAT文件的分割数据分布")
    print("="*80)
    
    all_results = []
    
    for i, filepath in enumerate(file_paths, 1):
        filename = os.path.basename(filepath)
        print(f"\n分析文件 {i:2d}: {filename}")
        
        try:
            with h5py.File(filepath, 'r') as f:
                # 读取数据
                big_seg = np.array(f['big_seg'])        # (384, 336, 256)
                region = np.array(f['region'])          # (384, 336, 256)  
                region_seg = np.array(f['region_seg'])  # (1, n_voxels)
                seg_one_hot = np.array(f['seg_one_hot']) # (102, n_voxels)
                
                # 分析big_seg
                big_seg_unique = np.unique(big_seg)
                big_seg_min, big_seg_max = np.min(big_seg), np.max(big_seg)
                big_seg_nonzero = np.sum(big_seg > 0)
                
                # 分析region
                region_unique = np.unique(region)
                region_min, region_max = np.min(region), np.max(region)
                region_nonzero = np.sum(region > 0)
                
                # 分析region_seg
                region_seg_flat = region_seg.flatten()
                region_seg_unique = np.unique(region_seg_flat)
                region_seg_min, region_seg_max = np.min(region_seg_flat), np.max(region_seg_flat)
                
                # 分析seg_one_hot
                active_regions_from_onehot = np.where(np.any(seg_one_hot, axis=1))[0]
                
                result = {
                    'file_idx': i,
                    'filename': filename,
                    'big_seg_min': big_seg_min,
                    'big_seg_max': big_seg_max,
                    'big_seg_unique_count': len(big_seg_unique),
                    'big_seg_nonzero_voxels': big_seg_nonzero,
                    'region_min': region_min,
                    'region_max': region_max,
                    'region_unique_count': len(region_unique),
                    'region_nonzero_voxels': region_nonzero,
                    'region_seg_min': region_seg_min,
                    'region_seg_max': region_seg_max,
                    'region_seg_unique_count': len(region_seg_unique),
                    'region_seg_n_voxels': len(region_seg_flat),
                    'seg_one_hot_active_regions': len(active_regions_from_onehot),
                    'big_seg_unique_first10': big_seg_unique[:10].tolist(),
                    'region_unique_first10': region_unique[:10].tolist(),
                    'region_seg_unique_first10': region_seg_unique[:10].tolist()
                }
                
                all_results.append(result)
                
                print(f"  big_seg: 范围 {big_seg_min}-{big_seg_max}, "
                      f"{len(big_seg_unique)}个唯一值, "
                      f"{big_seg_nonzero:,}个非零体素")
                print(f"  region:  范围 {region_min}-{region_max}, "
                      f"{len(region_unique)}个唯一值, "
                      f"{region_nonzero:,}个非零体素")
                print(f"  region_seg: 范围 {region_seg_min}-{region_seg_max}, "
                      f"{len(region_seg_unique)}个唯一值, "
                      f"{len(region_seg_flat):,}个体素")
                print(f"  seg_one_hot: {len(active_regions_from_onehot)}个激活的region")
                
                # 检查是否符合0-101分布
                is_big_seg_0_101 = big_seg_min == 0 and big_seg_max <= 101
                is_region_0_101 = region_min == 0 and region_max <= 101
                is_region_seg_0_101 = region_seg_min >= 0 and region_seg_max <= 101
                
                print(f"  是否符合0-101分布: big_seg={is_big_seg_0_101}, "
                      f"region={is_region_0_101}, region_seg={is_region_seg_0_101}")
                
        except Exception as e:
            print(f"  读取文件出错: {e}")
    
    # 创建汇总表格
    df = pd.DataFrame(all_results)
    
    print(f"\n{'='*80}")
    print("汇总统计:")
    print(f"{'='*80}")
    
    print(f"big_seg分析:")
    print(f"  最小值范围: {df['big_seg_min'].min()} - {df['big_seg_min'].max()}")
    print(f"  最大值范围: {df['big_seg_max'].min()} - {df['big_seg_max'].max()}")
    print(f"  唯一值数量范围: {df['big_seg_unique_count'].min()} - {df['big_seg_unique_count'].max()}")
    print(f"  所有文件是否都从0开始: {all(df['big_seg_min'] == 0)}")
    print(f"  所有文件是否都<=101: {all(df['big_seg_max'] <= 101)}")
    
    print(f"\nregion分析:")
    print(f"  最小值范围: {df['region_min'].min()} - {df['region_min'].max()}")
    print(f"  最大值范围: {df['region_max'].min()} - {df['region_max'].max()}")
    print(f"  唯一值数量范围: {df['region_unique_count'].min()} - {df['region_unique_count'].max()}")
    print(f"  所有文件是否都从0开始: {all(df['region_min'] == 0)}")
    print(f"  所有文件是否都<=101: {all(df['region_max'] <= 101)}")
    
    print(f"\nregion_seg分析:")
    print(f"  最小值范围: {df['region_seg_min'].min()} - {df['region_seg_min'].max()}")
    print(f"  最大值范围: {df['region_seg_max'].min()} - {df['region_seg_max'].max()}")
    print(f"  唯一值数量范围: {df['region_seg_unique_count'].min()} - {df['region_seg_unique_count'].max()}")
    print(f"  所有文件是否都>=0: {all(df['region_seg_min'] >= 0)}")
    print(f"  所有文件是否都<=101: {all(df['region_seg_max'] <= 101)}")
    
    print(f"\nseg_one_hot分析:")
    print(f"  激活region数量范围: {df['seg_one_hot_active_regions'].min()} - {df['seg_one_hot_active_regions'].max()}")
    
    return df

def analyze_detailed_distribution(sample_files=3):
    """
    详细分析前几个文件的数值分布
    """
    
    print(f"\n{'='*80}")
    print(f"详细分析前{sample_files}个文件的数值分布:")
    print(f"{'='*80}")
    
    file_paths = [
        "/home/jannik/Documents/mri_mat_onehot/OHC_13_ckgulxe.mat",
        "/home/jannik/Documents/mri_mat_onehot/OHC_16_degkbxy.mat",
        "/home/jannik/Documents/mri_mat_onehot/YHC_04_nnvuqjv.mat"
    ]
    
    for i, filepath in enumerate(file_paths[:sample_files], 1):
        filename = os.path.basename(filepath)
        print(f"\n详细分析文件 {i}: {filename}")
        
        with h5py.File(filepath, 'r') as f:
            big_seg = np.array(f['big_seg'])
            region = np.array(f['region'])
            region_seg = np.array(f['region_seg']).flatten()
            
            # 详细分布分析
            print(f"  big_seg唯一值: {sorted(np.unique(big_seg))}")
            print(f"  region唯一值: {sorted(np.unique(region))}")
            print(f"  region_seg唯一值: {sorted(np.unique(region_seg))}")
            
            # 检查是否有缺失的值
            big_seg_unique = set(np.unique(big_seg))
            region_unique = set(np.unique(region))
            region_seg_unique = set(np.unique(region_seg))
            
            expected_0_101 = set(range(102))  # 0-101
            
            print(f"  big_seg缺失的值(0-101): {sorted(expected_0_101 - big_seg_unique)}")
            print(f"  region缺失的值(0-101): {sorted(expected_0_101 - region_unique)}")
            print(f"  region_seg缺失的值(0-101): {sorted(expected_0_101 - region_seg_unique)}")
            
            # 统计各个值的数量
            big_seg_counts = Counter(big_seg.flatten())
            region_counts = Counter(region.flatten())
            region_seg_counts = Counter(region_seg)
            
            print(f"  big_seg中0的数量: {big_seg_counts[0]:,}")
            print(f"  region中0的数量: {region_counts[0]:,}")
            print(f"  region_seg中各值数量(前10): {dict(list(region_seg_counts.most_common(10)))}")

if __name__ == "__main__":
    # 执行分析
    df_results = analyze_segmentation_distributions()
    
    # 详细分析
    analyze_detailed_distribution(sample_files=3)
    
    print(f"\n分析完成！")

分析38个MAT文件的分割数据分布

分析文件  1: OHC_13_ckgulxe.mat
  big_seg: 范围 0.0-5002.0, 195个唯一值, 3,864,703个非零体素
  region:  范围 0-1, 2个唯一值, 2,069,287个非零体素
  region_seg: 范围 4.0-5002.0, 186个唯一值, 2,069,287个体素
  seg_one_hot: 99个激活的region
  是否符合0-101分布: big_seg=False, region=True, region_seg=False

分析文件  2: OHC_16_degkbxy.mat
  big_seg: 范围 0.0-5002.0, 194个唯一值, 4,076,566个非零体素
  region:  范围 0-1, 2个唯一值, 1,850,427个非零体素
  region_seg: 范围 4.0-5002.0, 179个唯一值, 1,850,427个体素
  seg_one_hot: 97个激活的region
  是否符合0-101分布: big_seg=False, region=True, region_seg=False

分析文件  3: YHC_04_nnvuqjv.mat
  big_seg: 范围 0.0-5002.0, 195个唯一值, 4,103,306个非零体素
  region:  范围 0-1, 2个唯一值, 2,105,627个非零体素
  region_seg: 范围 4.0-5002.0, 183个唯一值, 2,105,627个体素
  seg_one_hot: 97个激活的region
  是否符合0-101分布: big_seg=False, region=True, region_seg=False

分析文件  4: OHC_07_xhttjob.mat
  big_seg: 范围 0.0-5002.0, 194个唯一值, 4,611,418个非零体素
  region:  范围 0-1, 2个唯一值, 2,043,171个非零体素
  region_seg: 范围 4.0-5002.0, 180个唯一值, 2,043,171个体素
  seg_one_hot: 95个激活的region
  是否符合

In [12]:
import h5py
import numpy as np
import os

def strict_pixel_verification(filepath, check_first_n=None):
    """
    严格验证每个像素的映射关系，逐个检查前N个像素
    """
    filename = os.path.basename(filepath)
    print(f"\n严格验证文件: {filename}")
    print("="*60)
    
    with h5py.File(filepath, 'r') as f:
        big_seg = np.array(f['big_seg'])        # (384, 336, 256)
        region = np.array(f['region'])          # (384, 336, 256)
        region_seg = np.array(f['region_seg'])  # (1, n_voxels)
        
    region_seg_flat = region_seg.flatten()
    
    print(f"数据基本信息:")
    print(f"  3D shape: {big_seg.shape}")
    print(f"  region_seg length: {len(region_seg_flat)}")
    
    # 方法1: 直接使用np.where的默认顺序 (C-order)
    print(f"\n方法1: 使用np.where默认顺序 (C-order)")
    valid_indices_c = np.where(region == 1)
    big_seg_extracted_c = big_seg[valid_indices_c]
    
    print(f"  提取的体素数: {len(big_seg_extracted_c)}")
    print(f"  region_seg长度: {len(region_seg_flat)}")
    print(f"  数量匹配: {len(big_seg_extracted_c) == len(region_seg_flat)}")
    
    if len(big_seg_extracted_c) == len(region_seg_flat):
        # 检查100%像素（如果check_first_n为None）或指定数量
        if check_first_n is None:
            n_check = len(big_seg_extracted_c)
            print(f"  检查全部{n_check:,}个像素:")
        else:
            n_check = min(check_first_n, len(big_seg_extracted_c))
            print(f"  检查前{n_check:,}个像素:")
            
        mismatches = 0
        
        for i in range(n_check):
            if big_seg_extracted_c[i] != region_seg_flat[i]:
                mismatches += 1
                if mismatches <= 10:  # 只显示前10个不匹配
                    coord = (valid_indices_c[0][i], valid_indices_c[1][i], valid_indices_c[2][i])
                    print(f"    不匹配 {i}: 坐标{coord}, big_seg={big_seg_extracted_c[i]}, region_seg={region_seg_flat[i]}")
        
        print(f"  检查的{n_check:,}个像素中不匹配数: {mismatches}")
        
        # 如果没有检查全部，还要检查全部的统计
        if check_first_n is not None and n_check < len(big_seg_extracted_c):
            total_mismatches = np.sum(big_seg_extracted_c != region_seg_flat)
            print(f"  全部{len(big_seg_extracted_c):,}个像素中不匹配数: {total_mismatches}")
        else:
            total_mismatches = mismatches
        print(f"  全部{len(big_seg_extracted_c)}个像素中不匹配数: {total_mismatches}")
        print(f"  匹配率: {(len(big_seg_extracted_c) - total_mismatches) / len(big_seg_extracted_c) * 100:.6f}%")
        
        c_order_perfect = (total_mismatches == 0)
    else:
        c_order_perfect = False
    
    # 方法2: 手动展平为Fortran顺序再查找
    print(f"\n方法2: 手动展平为Fortran顺序")
    big_seg_flat_f = big_seg.flatten(order='F')
    region_flat_f = region.flatten(order='F')
    
    valid_mask_f = (region_flat_f == 1)
    big_seg_extracted_f = big_seg_flat_f[valid_mask_f]
    
    print(f"  提取的体素数: {len(big_seg_extracted_f)}")
    print(f"  数量匹配: {len(big_seg_extracted_f) == len(region_seg_flat)}")
    
    if len(big_seg_extracted_f) == len(region_seg_flat):
        total_mismatches_f = np.sum(big_seg_extracted_f != region_seg_flat)
        print(f"  全部不匹配数: {total_mismatches_f}")
        print(f"  匹配率: {(len(big_seg_extracted_f) - total_mismatches_f) / len(big_seg_extracted_f) * 100:.6f}%")
        
        fortran_order_perfect = (total_mismatches_f == 0)
    else:
        fortran_order_perfect = False
    
    # 方法3: 检查是否存在其他映射模式
    print(f"\n方法3: 检查其他可能的映射模式")
    
    # 尝试逆序
    big_seg_extracted_c_reversed = big_seg_extracted_c[::-1]
    reverse_mismatches = np.sum(big_seg_extracted_c_reversed != region_seg_flat)
    print(f"  逆序匹配不匹配数: {reverse_mismatches}")
    
    # 检查前几个和后几个值
    print(f"\n详细对比前10个和后10个值:")
    print(f"  C-order前10个:    {big_seg_extracted_c[:10]}")
    print(f"  region_seg前10个: {region_seg_flat[:10]}")
    print(f"  C-order后10个:    {big_seg_extracted_c[-10:]}")
    print(f"  region_seg后10个: {region_seg_flat[-10:]}")
    
    if len(big_seg_extracted_f) == len(region_seg_flat):
        print(f"  Fortran前10个:    {big_seg_extracted_f[:10]}")
        print(f"  Fortran后10个:    {big_seg_extracted_f[-10:]}")
    
    # 总结
    print(f"\n验证总结:")
    print(f"  C-order完美匹配: {c_order_perfect}")
    print(f"  Fortran-order完美匹配: {fortran_order_perfect}")
    
    if c_order_perfect:
        print("  ✅ 确认使用C-order (行优先) 映射")
    elif fortran_order_perfect:
        print("  ✅ 确认使用Fortran-order (列优先) 映射")
    else:
        print("  ❌ 没有找到完美的映射关系")
    
    return c_order_perfect, fortran_order_perfect

def verify_coordinate_mapping(filepath, sample_size=100):
    """
    通过随机采样验证坐标映射的正确性
    """
    filename = os.path.basename(filepath)
    print(f"\n坐标映射验证: {filename}")
    print("="*50)
    
    with h5py.File(filepath, 'r') as f:
        big_seg = np.array(f['big_seg'])        
        region = np.array(f['region'])          
        region_seg = np.array(f['region_seg']).flatten()
        
    # 获取有效坐标
    valid_coords = np.where(region == 1)
    n_valid = len(valid_coords[0])
    
    # 随机采样
    sample_indices = np.random.choice(n_valid, min(sample_size, n_valid), replace=False)
    
    print(f"随机检查{len(sample_indices)}个坐标的映射:")
    
    mismatches = 0
    for i, idx in enumerate(sample_indices):
        x, y, z = valid_coords[0][idx], valid_coords[1][idx], valid_coords[2][idx]
        big_seg_value = big_seg[x, y, z]
        region_seg_value = region_seg[idx]
        
        if big_seg_value != region_seg_value:
            mismatches += 1
            if mismatches <= 5:  # 只显示前5个不匹配
                print(f"  不匹配 {i}: 坐标({x},{y},{z}), big_seg={big_seg_value}, region_seg={region_seg_value}")
    
    match_rate = (len(sample_indices) - mismatches) / len(sample_indices) * 100
    print(f"随机采样匹配率: {match_rate:.2f}% ({len(sample_indices) - mismatches}/{len(sample_indices)})")
    
    return match_rate == 100.0

if __name__ == "__main__":
    test_file = "/home/jannik/Documents/mri_mat_onehot/OHC_13_ckgulxe.mat"
    
    print("严格的逐像素验证")
    print("="*80)
    
    # 严格验证 - 检查100%像素
    c_perfect, f_perfect = strict_pixel_verification(test_file, check_first_n=None)
    
    # 坐标映射验证
    coord_perfect = verify_coordinate_mapping(test_file, sample_size=1000)
    
    print(f"\n最终结论:")
    print(f"="*40)
    if c_perfect and coord_perfect:
        print("✅ 确认映射关系：region=1的体素按C-order (行优先) 顺序对应region_seg")
    elif f_perfect and coord_perfect:
        print("✅ 确认映射关系：region=1的体素按Fortran-order (列优先) 顺序对应region_seg")  
    else:
        print("❌ 映射关系验证失败，需要进一步分析")

严格的逐像素验证

严格验证文件: OHC_13_ckgulxe.mat
数据基本信息:
  3D shape: (384, 336, 256)
  region_seg length: 2069287

方法1: 使用np.where默认顺序 (C-order)
  提取的体素数: 2069287
  region_seg长度: 2069287
  数量匹配: True
  检查全部2,069,287个像素:
  检查的2,069,287个像素中不匹配数: 0
  全部2069287个像素中不匹配数: 0
  匹配率: 100.000000%

方法2: 手动展平为Fortran顺序
  提取的体素数: 2069287
  数量匹配: True
  全部不匹配数: 2053157
  匹配率: 0.779496%

方法3: 检查其他可能的映射模式
  逆序匹配不匹配数: 2057984

详细对比前10个和后10个值:
  C-order前10个:    [1011. 1011. 1011. 1011. 1011. 1011. 1011. 1011. 1011. 1011.]
  region_seg前10个: [1011. 1011. 1011. 1011. 1011. 1011. 1011. 1011. 1011. 1011.]
  C-order后10个:    [2028. 2028. 2028. 2028. 2028. 2028. 2028. 1028. 1028. 1028.]
  region_seg后10个: [2028. 2028. 2028. 2028. 2028. 2028. 2028. 1028. 1028. 1028.]
  Fortran前10个:    [1030. 1030. 1030. 1015. 1015. 1015. 1015. 1015. 1015. 1015.]
  Fortran后10个:    [2015. 2015. 2015. 2015. 2015. 2015. 2015. 2015. 2015. 2015.]

验证总结:
  C-order完美匹配: True
  Fortran-order完美匹配: False
  ✅ 确认使用C-order (行优先) 映射

坐标映射验证: OHC_13_ckgulxe

In [13]:
import h5py
import numpy as np

def matlab_revert_reshape(array_1d, region_mask):
    """
    Python实现的MATLAB revert_reshape函数
    
    MATLAB原代码逻辑:
    big_img = single(zeros([size(region) size(array,2)]));
    for ii = 1:1:size(array,2)
        img = big_img(:,:,:,ii);
        img_index = img(:);                    % 展平为1D (Fortran order)
        template_index = region(:);            % region展平为1D (Fortran order)  
        img_index(template_index) = array(:,ii);  % 在template_index为真的位置填入数据
        img = reshape(img_index, size(region));   % 重新变为3D
        big_img(:,:,:,ii) = img;
    end
    """
    
    h, w, d = region_mask.shape
    n_voxels, n_features = array_1d.shape
    
    # 创建输出数组
    big_img = np.zeros((h, w, d, n_features), dtype=np.float32)
    
    for i in range(n_features):
        # 创建临时3D数组
        img = np.zeros((h, w, d))
        
        # 关键：MATLAB使用Fortran顺序展平和重塑
        img_flat = img.flatten(order='F')  # MATLAB的(:)操作
        region_flat = region_mask.flatten(order='F')  # MATLAB的(:)操作
        
        # 在region为真（非零）的位置填入数据
        # MATLAB中逻辑索引：template_index为region的逻辑值
        valid_positions = (region_flat > 0)  # 或者 (region_flat == 1)
        
        if np.sum(valid_positions) != n_voxels:
            print(f"警告：有效位置数量({np.sum(valid_positions)})与数据长度({n_voxels})不匹配")
        
        img_flat[valid_positions] = array_1d[:, i]
        
        # 重新变为3D (Fortran顺序)
        img = img_flat.reshape((h, w, d), order='F')
        big_img[:, :, :, i] = img
    
    return big_img

def test_revert_reshape_mapping():
    """
    测试revert_reshape函数是否与我们验证的映射一致
    """
    
    filepath = "/home/jannik/Documents/mri_mat_onehot/OHC_13_ckgulxe.mat"
    
    print("测试revert_reshape函数的映射关系")
    print("="*60)
    
    with h5py.File(filepath, 'r') as f:
        big_seg = np.array(f['big_seg'])        # (384, 336, 256)
        region = np.array(f['region'])          # (384, 336, 256)
        region_seg = np.array(f['region_seg'])  # (1, n_voxels)
        multidim_data = np.array(f['multidim_data'])  # (351, n_voxels)
    
    # 转置数据以匹配函数期望的格式
    region_seg_t = region_seg.T  # (n_voxels, 1)
    multidim_data_t = multidim_data.T  # (n_voxels, 351)
    
    print(f"数据形状:")
    print(f"  region: {region.shape}")
    print(f"  region_seg转置后: {region_seg_t.shape}")
    print(f"  multidim_data转置后: {multidim_data_t.shape}")
    
    # 方法1: 使用MATLAB风格的revert_reshape
    print(f"\n方法1: 使用MATLAB风格的revert_reshape (Fortran顺序)")
    try:
        reconstructed_matlab = matlab_revert_reshape(region_seg_t, region)
        print(f"  重构成功，形状: {reconstructed_matlab.shape}")
        
        # 验证重构是否正确：检查重构的数据是否与原始big_seg在有效位置匹配
        valid_mask = (region > 0)
        original_values = big_seg[valid_mask]
        reconstructed_values = reconstructed_matlab[:, :, :, 0][valid_mask]
        
        perfect_match = np.array_equal(original_values, reconstructed_values)
        print(f"  重构数据与原始big_seg完美匹配: {perfect_match}")
        
        if not perfect_match:
            mismatch_count = np.sum(original_values != reconstructed_values)
            match_rate = (len(original_values) - mismatch_count) / len(original_values) * 100
            print(f"  匹配率: {match_rate:.6f}%")
            
            # 显示一些不匹配的例子
            if mismatch_count > 0:
                diff_mask = original_values != reconstructed_values
                print(f"  前5个不匹配: 原始={original_values[diff_mask][:5]}, 重构={reconstructed_values[diff_mask][:5]}")
        
    except Exception as e:
        print(f"  MATLAB风格重构失败: {e}")
        perfect_match = False
    
    # 方法2: 使用我们验证过的C-order方式
    print(f"\n方法2: 使用验证过的C-order方式")
    try:
        big_img_c = np.zeros(region.shape + (1,), dtype=np.float32)
        valid_mask = (region > 0)
        big_img_c[valid_mask, 0] = region_seg.flatten()
        
        # 验证
        c_perfect_match = np.array_equal(big_seg[valid_mask], big_img_c[valid_mask, 0])
        print(f"  C-order重构与原始big_seg完美匹配: {c_perfect_match}")
        
    except Exception as e:
        print(f"  C-order重构失败: {e}")
        c_perfect_match = False
    
    # 对比两种方法
    print(f"\n结论:")
    if perfect_match:
        print("✅ MATLAB的revert_reshape函数使用Fortran顺序，与数据一致")
        print("   这意味着原始数据生成时也使用了Fortran顺序")
    elif c_perfect_match:
        print("✅ 数据使用C-order顺序，与我们之前的验证一致")
        print("   MATLAB的revert_reshape函数可能需要调整顺序")
    else:
        print("❌ 两种方法都不能完美重构，需要进一步分析")
    
    return perfect_match, c_perfect_match

def analyze_matlab_indexing():
    """
    分析MATLAB索引方式与Python的差异
    """
    
    print(f"\n分析MATLAB与Python的索引差异:")
    print("="*50)
    
    # 创建小型测试数据
    test_region = np.array([[[0, 1], [1, 0]], [[1, 1], [0, 1]]])  # (2,2,2)
    print(f"测试region形状: {test_region.shape}")
    print(f"测试region:\n{test_region}")
    
    # Python方式 (C-order)
    valid_mask_c = (test_region > 0)
    valid_coords_c = np.where(valid_mask_c)
    print(f"\nPython C-order有效坐标:")
    for i, (x, y, z) in enumerate(zip(*valid_coords_c)):
        print(f"  {i}: ({x},{y},{z}) = {test_region[x,y,z]}")
    
    # MATLAB方式 (Fortran-order)
    region_flat_f = test_region.flatten(order='F')
    valid_indices_f = np.where(region_flat_f > 0)[0]
    print(f"\nMATLAB Fortran-order展平:")
    print(f"  展平数组: {region_flat_f}")
    print(f"  有效索引: {valid_indices_f}")
    
    # 转换回3D坐标
    print(f"  对应的3D坐标:")
    for i, flat_idx in enumerate(valid_indices_f):
        # Fortran顺序的索引转换
        x = flat_idx % test_region.shape[0]
        y = (flat_idx // test_region.shape[0]) % test_region.shape[1]
        z = flat_idx // (test_region.shape[0] * test_region.shape[1])
        print(f"    {i}: 展平索引{flat_idx} -> 3D({x},{y},{z}) = {test_region[x,y,z]}")

if __name__ == "__main__":
    # 测试revert_reshape
    matlab_match, c_match = test_revert_reshape_mapping()
    
    # 分析索引差异
    analyze_matlab_indexing()
    
    print(f"\n最终结论:")
    print("="*40)
    if matlab_match and not c_match:
        print("数据生成使用Fortran顺序，应该使用MATLAB的revert_reshape函数")
    elif c_match and not matlab_match:
        print("数据生成使用C顺序，需要调整revert_reshape函数或使用Python版本")
    elif matlab_match and c_match:
        print("两种方法都可以，但这不太可能...")
    else:
        print("需要进一步分析数据的生成方式")

测试revert_reshape函数的映射关系
数据形状:
  region: (384, 336, 256)
  region_seg转置后: (2069287, 1)
  multidim_data转置后: (2069287, 351)

方法1: 使用MATLAB风格的revert_reshape (Fortran顺序)
  重构成功，形状: (384, 336, 256, 1)
  重构数据与原始big_seg完美匹配: False
  匹配率: 0.779496%
  前5个不匹配: 原始=[1011. 1011. 1011. 1011. 1011.], 重构=[  17. 5001. 2012. 1030. 4011.]

方法2: 使用验证过的C-order方式
  C-order重构与原始big_seg完美匹配: True

结论:
✅ 数据使用C-order顺序，与我们之前的验证一致
   MATLAB的revert_reshape函数可能需要调整顺序

分析MATLAB与Python的索引差异:
测试region形状: (2, 2, 2)
测试region:
[[[0 1]
  [1 0]]

 [[1 1]
  [0 1]]]

Python C-order有效坐标:
  0: (0,0,1) = 1
  1: (0,1,0) = 1
  2: (1,0,0) = 1
  3: (1,0,1) = 1
  4: (1,1,1) = 1

MATLAB Fortran-order展平:
  展平数组: [0 1 1 0 1 1 0 1]
  有效索引: [1 2 4 5 7]
  对应的3D坐标:
    0: 展平索引1 -> 3D(1,0,0) = 1
    1: 展平索引2 -> 3D(0,1,0) = 1
    2: 展平索引4 -> 3D(0,0,1) = 1
    3: 展平索引5 -> 3D(1,0,1) = 1
    4: 展平索引7 -> 3D(1,1,1) = 1

最终结论:
数据生成使用C顺序，需要调整revert_reshape函数或使用Python版本
